In [0]:
# Databricks notebook source
# =============================================================
# Unit tests — bronze photo_raw
# Dummy data simulates: 1 good row, 1 bad row (rescued), 1 null url, 1 duplicate
# =============================================================

from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField,
    LongType, StringType, DoubleType, TimestampType,
)
from datetime import datetime

NOW = datetime.now()

# -------------------------------------------------------------
# Dummy dataset — 4 rows, each tests something different
# -------------------------------------------------------------
# Row 1: good row — everything valid
# Row 2: good row — duplicate of row 1 (simulates COPY INTO running twice)
# Row 3: bad row — has extra column, Spark rescued it into _rescued_data
# Row 4: bad row — photo_url is NULL (missing in CSV)

SCHEMA = StructType([
    StructField("photo_seq_id",    LongType(),      True),
    StructField("photo_url",       StringType(),    True),
    StructField("id",              DoubleType(),    True),
    StructField("load_dt",         TimestampType(), True),
    StructField("modification_dt", TimestampType(), True),
    StructField("source",          StringType(),    True),
    StructField("_rescued_data",   StringType(),    True),
])

dummy_data = [
    Row(101, "https://img.com/photo_101.jpg", 500.5, NOW, NOW, "/path/1_photo.csv", None),                          # good
    Row(101, "https://img.com/photo_101.jpg", 500.5, NOW, NOW, "/path/1_photo.csv", None),                          # duplicate of row 1
    Row(202, "https://img.com/photo_202.jpg", 600.0, NOW, NOW, "/path/1_photo.csv", '{"oops_extra_column":"bad"}'), # rescued
    Row(303, None,                            700.0, NOW, NOW, "/path/1_photo.csv", None),                          # null url
]

df = spark.createDataFrame(dummy_data, SCHEMA)


# -------------------------------------------------------------
# Tests
# -------------------------------------------------------------

# --- TEST 1: Record count ---
total_count = df.count()
assert total_count == 4, f"TEST FAILED: Expected 4 rows, found {total_count}"
print("✅ Test 1 Passed: Total Record Count")

# --- TEST 2: Good data parsing ---
good_record = df.filter("photo_seq_id = 101").first()
assert good_record["id"] == 500.5, "TEST FAILED: Data type mismatch or parsing error on ID"
assert good_record["_rescued_data"] is None or good_record["_rescued_data"] == "", "TEST FAILED: Good record was falsely rescued"
print("✅ Test 2 Passed: Data Parsing")

# --- TEST 3: Rescue logic ---
bad_record = df.filter("photo_seq_id = 202").first()
assert bad_record["_rescued_data"] is not None, "TEST FAILED: Malformed row was NOT rescued"
assert "oops_extra_column" in (bad_record["_rescued_data"] or ""), "TEST FAILED: Rescued data content missing"
print("✅ Test 3 Passed: Rescue Logic")

# --- TEST 4: Null URL detected ---
null_url_record = df.filter("photo_seq_id = 303").first()
assert null_url_record["photo_url"] is None, "TEST FAILED: Expected NULL photo_url on row 303"
print("✅ Test 4 Passed: Null URL Detected")

# --- TEST 5: Duplicate detected ---
dup_count = df.filter("photo_seq_id = 101").count()
assert dup_count == 2, f"TEST FAILED: Expected 2 duplicates for seq_id 101, found {dup_count}"
print("✅ Test 5 Passed: Duplicate Detected")